In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import numpy as np
from scipy.stats import ttest_ind, ttest_rel, wilcoxon, spearmanr, pearsonr

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

In [7]:
# Load your sentiment data
portfolio_tickers_sentiment = pd.read_csv('v2_04.csv').round(2)

print(f"   Data Overview:")
print(f"   Shape: {portfolio_tickers_sentiment.shape}")
print(f"   Columns: {list(portfolio_tickers_sentiment.columns)}")
print(f"   Unique tickers: {len(portfolio_tickers_sentiment['ticker'].unique())}")
print(f"   Time range: {portfolio_tickers_sentiment['year'].min()} to {portfolio_tickers_sentiment['year'].max()}")

# Display first few rows
print(f"\n Sample data:")
portfolio_tickers_sentiment

   Data Overview:
   Shape: (1095, 11)
   Columns: ['ticker', 'year', 'month', 'article_count', 'tone_mean', 'positive_mean', 'negative_mean', 'sentiment_balance', 'confidence_level', 'sample_urls', 'sources_used']
   Unique tickers: 12
   Time range: 2015 to 2025

 Sample data:


,ticker,year,month,article_count,tone_mean,positive_mean,negative_mean,sentiment_balance,confidence_level,sample_urls,sources_used
0,AMD,2015,3,21,0.61,2.81,2.20,0.01,medium,[http://www.nasdaq.com/article/tech-stocks-2-t...,"[finance.yahoo.com,nasdaq.com,marketwatch.com,..."
1,AMD,2015,4,40,0.23,3.05,2.82,0.00,medium,[http://www.nasdaq.com/article/european-stocks...,"[bloomberg.com,marketwatch.com,forbes.com,reut..."
2,AMD,2015,5,26,1.11,2.83,1.72,0.01,medium,[http://www.nytimes.com/2015/05/08/business/de...,"[nasdaq.com,marketwatch.com,forbes.com,reuters..."
3,AMD,2015,6,42,1.00,3.16,2.16,0.01,medium,[http://www.forbes.com/sites/jasonevangelho/20...,"[finance.yahoo.com,forbes.com,reuters.com,nasd..."
4,AMD,2015,7,73,-0.69,2.46,3.15,-0.01,high,[http://www.nasdaq.com/article/amd-september-4...,"[morningstar.com,marketwatch.com,nytimes.com,r..."
...,...,...,...,...,...,...,...,...,...,...,...
1090,SPY,2025,4,30,-2.84,2.22,5.06,-0.03,medium,[https://www.yahoo.com/news/russian-spy-accuse...,"[finance.yahoo.com,forbes.com,yahoo.com]"
1091,SPY,2025,5,46,-2.36,2.48,4.84,-0.02,medium,[https://www.yahoo.com/news/german-spy-agency-...,"[cnbc.com,forbes.com,yahoo.com,bloomberg.com,f..."
1092,SPY,2025,6,17,-1.81,2.37,4.17,-0.02,low,[https://www.yahoo.com/news/star-wars-simpsons...,"[yahoo.com,finance.yahoo.com,forbes.com,econom..."
1093,SPY,2025,7,42,-2.71,2.36,5.07,-0.03,medium,[https://www.businessinsider.com/military-expe...,"[bloomberg.com,yahoo.com,finance.yahoo.com,bus..."


In [8]:
from utile import *
# Test the function with your sentiment data
coverage_analysis = analyze_data_coverage(portfolio_tickers_sentiment)
coverage_analysis[['ticker', 'coverage_percentage_per_existed', 'total_articles']]


Data range: 2015-02 to 2025-08
Expected total months in range: 127

Data Coverage by Ticker:


,ticker,coverage_percentage_per_existed,total_articles
0,GOOGL,100.0,284511
1,MSFT,100.0,163344
2,MU,100.0,10331
3,NVDA,100.0,65874
4,AMD,98.4,9184
5,SPY,96.1,6723
6,MRVL,74.8,2567
7,RDDT,63.0,2175
8,PLTR,58.6,2283
9,QQQ,38.9,2122


In [9]:
# Test the function with your sentiment data
missing_months = analyze_missing_months_detailed(portfolio_tickers_sentiment)
missing_months

Total missing month-ticker combinations: 335
Sample missing data:


,ticker,year,month,year_month
0,AMD,2015,12,2015-12
1,AMD,2016,2,2016-02
2,APP,2021,5,2021-05
3,APP,2021,6,2021-06
4,APP,2021,7,2021-07
...,...,...,...,...
330,SPY,2021,8,2021-08
331,SPY,2022,1,2022-01
332,SPY,2022,4,2022-04
333,SPY,2022,5,2022-05


In [8]:
missing_months.groupby('ticker').agg({'year_month': 'count'})

,year_month
ticker,
AMD,2
APP,40
ASML,88
MRVL,28
PLTR,33
QQQ,2
RDDT,63
SPY,11


In [9]:
portfolio_tickers_sentiment[portfolio_tickers_sentiment['ticker'] == 'SPY']

,ticker,year,month,article_count,tone_mean,positive_mean,negative_mean,sentiment_balance,confidence_level,sample_urls,sources_used
606,SPY,2015,2,30,-2.43,2.16,4.59,-0.02,medium,[http://www.businessinsider.com/edward-snowden...,"[finance.yahoo.com,nytimes.com,economist.com,b..."
607,SPY,2015,3,56,-2.52,2.35,4.87,-0.02,high,[http://www.wsj.com/articles/the-10-point-gera...,"[forbes.com,businessinsider.com,wsj.com,nasdaq..."
608,SPY,2015,4,43,-2.63,2.05,4.68,-0.03,medium,[http://www.nytimes.com/aponline/2015/04/30/wo...,"[forbes.com,nasdaq.com,washingtonpost.com,fina..."
609,SPY,2015,5,71,-2.31,2.28,4.59,-0.02,high,[http://www.businessinsider.com/afp-germanys-m...,"[economist.com,forbes.com,bloomberg.com,nytime..."
610,SPY,2015,6,78,-1.95,2.72,4.67,-0.02,high,[http://www.businessinsider.com/r-rwanda-says-...,"[bloomberg.com,nytimes.com,forbes.com,washingt..."
...,...,...,...,...,...,...,...,...,...,...,...
723,SPY,2025,4,30,-2.84,2.22,5.06,-0.03,medium,[https://finance.yahoo.com/news/spy-attracts-5...,"[forbes.com,yahoo.com,finance.yahoo.com]"
724,SPY,2025,5,46,-2.36,2.48,4.84,-0.02,medium,[https://finance.yahoo.com/news/spy-adds-2b-ma...,"[yahoo.com,finance.yahoo.com,forbes.com,cnbc.c..."
725,SPY,2025,6,17,-1.81,2.37,4.17,-0.02,low,[https://finance.yahoo.com/news/spy-attracts-1...,"[finance.yahoo.com,yahoo.com,forbes.com,econom..."
726,SPY,2025,7,42,-2.71,2.36,5.07,-0.03,medium,[https://www.yahoo.com/news/british-man-guilty...,"[forbes.com,businessinsider.com,finance.yahoo...."
